# 01 · Prepare text and image features (Phases 3 and 4)

This notebook runs **once**. It turns the raw data into numbers that the models can learn from, and saves them so every later notebook can reuse them.

| Part | What happens | Saved file |
|---|---|---|
| Phase 3: text | clean captions, split into words, build vocabularies, turn captions into lists of numbers | `captions_split.csv`, `vocab_en.json`, `vocab_bn.json`, `encoded_en.npz`, `encoded_bn.npz` |
| Phase 4: images | run a pretrained ResNet-50 over all 8,091 photos and keep its output | `images.json`, `feats_pooled.npy`, `feats_grid.npy` |

**On Kaggle:** add the inputs `adityajn105/flickr8k` and `saifsust/bancap`, set **Accelerator: GPU T4** and **Internet: On**, then run all cells (about 10 to 15 minutes).
When it works, click **Save Version → Save & Run All (Commit)** so the output files are stored. Notebook 02 reads them from there.

## Setup

In [9]:
import sys;
print(sys.executable)

/home/sakib/projects/image_captioning/.venv/bin/python


In [10]:
import os, re, json, io, zipfile, random, time, unicodedata, urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from PIL import Image

SEED = 42                                   # same seed = same results every run
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

PyTorch 2.15.0+cpu | device: cpu 


## Find the input files
On Kaggle the data sits under `/kaggle/input`. On your PC it is in the project's `data/` folder. The cell searches for the files by name, so the exact folder layout does not matter.

In [11]:
ON_KAGGLE = Path("/kaggle/input").exists()
ROOTS = [Path("/kaggle/input")] if ON_KAGGLE else [Path("../data"), Path("data")]
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

def find(name, want_dir=False):
    for root in ROOTS:
        if not root.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if want_dir and name in dirnames:
                return Path(dirpath) / name
            if not want_dir and name in filenames:
                return Path(dirpath) / name
    return None

IMG_DIR    = find("Images", want_dir=True)
BANCAP_CSV = find("BAN-Cap_captiondata.csv")
SPLIT_TRAIN = find("Flickr_8k.trainImages.txt")

# the official split files: use local copies if present, otherwise download them (needs Internet: On)
if SPLIT_TRAIN is None:
    url = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip"
    split_dir = OUT / "flickr8k_splits"
    split_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())) as z:
        for n in ["Flickr_8k.trainImages.txt", "Flickr_8k.devImages.txt", "Flickr_8k.testImages.txt"]:
            (split_dir / n).write_bytes(z.read(n))
    SPLIT_TRAIN = split_dir / "Flickr_8k.trainImages.txt"
SPLIT_DIR = SPLIT_TRAIN.parent

for k, v in {"images": IMG_DIR, "BAN-Cap csv": BANCAP_CSV, "split files": SPLIT_DIR, "output folder": OUT}.items():
    print(f"{k:14s} {v}")
assert IMG_DIR and BANCAP_CSV, "Add the flickr8k and bancap datasets as inputs"

images         ../data/Images
BAN-Cap csv    ../data/BAN-Cap_captiondata.csv
split files    ../data/flickr8k_splits
output folder  ../data/processed


## Load captions and apply the frozen split
Same rules as Phase 2: BAN-Cap gives English and Bangla captions side by side, and the official split puts 6,000 / 1,000 / 1,000 images in train / validation / test.

In [12]:
df = pd.read_csv(BANCAP_CSV, encoding="utf-8")
df["image"]   = df["caption_id"].str.split("#").str[0]
df["cap_idx"] = df["caption_id"].str.split("#").str[1].astype(int)
df = df.rename(columns={"english_caption": "en", "bengali_caption": "bn"})[["image", "cap_idx", "en", "bn"]]

def read_list(name):
    return [l.strip() for l in open(SPLIT_DIR / name, encoding="utf-8") if l.strip()]
split_of = {}
for split, name in [("train", "Flickr_8k.trainImages.txt"), ("val", "Flickr_8k.devImages.txt"), ("test", "Flickr_8k.testImages.txt")]:
    for img in read_list(name):
        split_of[img] = split
df["split"] = df["image"].map(split_of).fillna("unused")

print(df.groupby("split").image.nunique().to_dict())
assert df[df.split == "train"].image.nunique() == 6000

{'test': 1000, 'train': 6000, 'unused': 91, 'val': 1000}


## Phase 3 · Text preprocessing

A neural network only understands numbers, so every caption becomes a list of word numbers.

1. **Tokenise:** split a caption into words.
   * English: lowercase, keep only letters.
   * Bangla: Unicode NFC normalisation (one standard form for letters that can be typed two ways), then remove punctuation such as the dari `।`.
2. **Vocabulary:** the list of words the model may write. We keep words that appear at least **5 times in the training captions**. Rarer words become `<unk>` (unknown). Using only training captions avoids data leakage.
3. **Special tokens:** `<pad>` = 0 fills empty space, `<start>` = 1 begins every caption, `<end>` = 2 ends it, `<unk>` = 3 replaces rare words.
4. **Padding:** all captions get the same length, so they can be stacked into one table (a tensor). Very long captions are cut.

In [13]:
BN_PUNCT = re.compile(r"[।॥|.,!?;:\"'()\[\]{}\-–—/\\#]")

def tok_en(s):
    return re.findall(r"[a-z]+", str(s).lower())

def tok_bn(s):
    s = unicodedata.normalize("NFC", str(s))
    return BN_PUNCT.sub(" ", s).split()

SPECIALS = ["<pad>", "<start>", "<end>", "<unk>"]
PAD, START, END, UNK = 0, 1, 2, 3
MIN_FREQ = 5
MAX_WORDS = {"en": 20, "bn": 16}     # longer captions are cut (about 98% of captions fit)

def build_vocab(token_lists, min_freq=MIN_FREQ):
    counts = Counter(w for toks in token_lists for w in toks)
    words = sorted([w for w, c in counts.items() if c >= min_freq], key=lambda w: (-counts[w], w))
    return SPECIALS + words

def encode(tokens, stoi, max_words):
    ids = [START] + [stoi.get(w, UNK) for w in tokens[:max_words]] + [END]
    return ids + [PAD] * (max_words + 2 - len(ids))

def decode(ids, itos):
    words = []
    for i in ids:
        if i == END: break
        if i not in (PAD, START): words.append(itos[i])
    return words

df["en_tok"] = df["en"].map(lambda s: " ".join(tok_en(s)))
df["bn_tok"] = df["bn"].map(lambda s: " ".join(tok_bn(s)))
train = df[df.split == "train"]

vocab = {}
for lang in ["en", "bn"]:
    itos = build_vocab(train[f"{lang}_tok"].str.split())
    vocab[lang] = itos
    lens = df[f"{lang}_tok"].str.split().map(len)
    covered = (lens <= MAX_WORDS[lang]).mean()
    unk_rate = np.mean([w not in set(itos) for toks in df[df.split == "val"][f"{lang}_tok"].str.split() for w in toks])
    print(f"{lang}: vocabulary {len(itos):5d} words | captions not cut: {covered:.1%} | unknown words in val: {unk_rate:.1%}")

en: vocabulary  2533 words | captions not cut: 98.5% | unknown words in val: 3.2%
bn: vocabulary  3110 words | captions not cut: 98.0% | unknown words in val: 6.9%


**Expected:** English vocabulary about 2,500 words, Bangla about 3,100. About 98% of captions fit without cutting. Unknown words in validation: about 3% for English and 7% for Bangla, because Bangla words change form more.

In [14]:
# encode every caption and check the round trip on one example
encoded = {}
for lang in ["en", "bn"]:
    stoi = {w: i for i, w in enumerate(vocab[lang])}
    encoded[lang] = np.array([encode(t.split(), stoi, MAX_WORDS[lang]) for t in df[f"{lang}_tok"]], dtype=np.int32)

row = df[df.split == "train"].iloc[0]
i = row.name
for lang in ["en", "bn"]:
    print(f"\n[{lang}] original : {row[lang]}")
    print(f"[{lang}] tokens   : {row[f'{lang}_tok'].split()}")
    print(f"[{lang}] ids      : {encoded[lang][i].tolist()}")
    print(f"[{lang}] decoded  : {' '.join(decode(encoded[lang][i], vocab[lang]))}")
print("\nencoded shapes:", {k: v.shape for k, v in encoded.items()})


[en] original : A child in a pink dress is climbing up a set of stairs in an entry way .
[en] tokens   : ['a', 'child', 'in', 'a', 'pink', 'dress', 'is', 'climbing', 'up', 'a', 'set', 'of', 'stairs', 'in', 'an', 'entry', 'way']
[en] ids      : [1, 4, 44, 5, 4, 89, 173, 8, 118, 54, 4, 393, 13, 400, 5, 29, 3, 641, 2, 0, 0, 0]
[en] decoded  : a child in a pink dress is climbing up a set of stairs in an <unk> way

[bn] original : একটি গোলাপী জামা পরা বাচ্চা মেয়ে একটি বাড়ির প্রবেশ পথের সিঁড়ি বেয়ে উঠছে।
[bn] tokens   : ['একটি', 'গোলাপী', 'জামা', 'পরা', 'বাচ্চা', 'মেয়ে', 'একটি', 'বাড়ির', 'প্রবেশ', 'পথের', 'সিঁড়ি', 'বেয়ে', 'উঠছে']
[bn] ids      : [1, 4, 176, 46, 27, 34, 21, 4, 424, 1224, 867, 649, 126, 137, 2, 0, 0, 0]
[bn] decoded  : একটি গোলাপী জামা পরা বাচ্চা মেয়ে একটি বাড়ির প্রবেশ পথের সিঁড়ি বেয়ে উঠছে

encoded shapes: {'en': (40455, 22), 'bn': (40455, 18)}


**What to notice:** every id list starts with `1` (`<start>`), ends with `2` (`<end>`) and is filled with `0` (`<pad>`). Decoding the ids gives the tokens back, which proves the encoding is correct. A rare word (here possibly "entry") comes back as `<unk>`, which is expected.

## Phase 4 · Image features with a pretrained CNN

**ResNet-50** is a convolutional neural network trained on 1.2 million ImageNet photos. We do not retrain it (it stays **frozen**). We cut off its final classification layer and keep what it "sees":

* **Grid features** `7 × 7 × 2048`: 49 image regions, each described by 2,048 numbers. The Transformer (Phase 7) will look at these regions.
* **Pooled features** `2048`: the average of the 49 regions, one summary vector per photo. The LSTM uses this.

Running the CNN takes most of the compute, so we do it **once** and save the results in half precision (float16) to halve the file size. All later training only reads these files, which is why training is fast.

In [15]:
images = sorted(df.image.unique())                    # fixed order: row i of the feature files = images[i]
img_index = {name: i for i, name in enumerate(images)}

tfm = transforms.Compose([
    transforms.Resize((224, 224)),                     # the size ResNet-50 was trained on
    transforms.ToTensor(),                             # pixels 0..255 -> 0..1
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),   # ImageNet mean / std
])

class ImageFolderList(torch.utils.data.Dataset):
    def __len__(self): return len(images)
    def __getitem__(self, i):
        with Image.open(IMG_DIR / images[i]) as im:
            return tfm(im.convert("RGB")), i

import multiprocessing as mp
ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
loader = torch.utils.data.DataLoader(ImageFolderList(), batch_size=64, shuffle=False,
                                     num_workers=2 if ctx else 0, multiprocessing_context=ctx,
                                     pin_memory=(DEVICE == "cuda"))

resnet = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
encoder = nn.Sequential(*list(resnet.children())[:-2]).to(DEVICE).eval()   # drop avgpool + fc
for p in encoder.parameters():
    p.requires_grad = False                            # frozen
print("encoder ready, parameters:", sum(p.numel() for p in encoder.parameters()) // 10**6, "million")

encoder ready, parameters: 23 million


In [16]:
N = len(images)
pooled = np.zeros((N, 2048), dtype=np.float16)
grid = np.lib.format.open_memmap(OUT / "feats_grid.npy", mode="w+", dtype=np.float16, shape=(N, 49, 2048))

t0 = time.time()
with torch.no_grad():
    for b, (x, idx) in enumerate(loader):
        fmap = encoder(x.to(DEVICE, non_blocking=True))          # (B, 2048, 7, 7)
        g = fmap.flatten(2).transpose(1, 2)                      # (B, 49, 2048)
        grid[idx.numpy()] = g.half().cpu().numpy()
        pooled[idx.numpy()] = g.mean(dim=1).half().cpu().numpy()
        if b % 20 == 0:
            print(f"  batch {b:3d}/{len(loader)}  {time.time() - t0:5.0f}s")
grid.flush(); del grid
np.save(OUT / "feats_pooled.npy", pooled)
print(f"done in {time.time() - t0:.0f}s")

  batch   0/127      3s
  batch  20/127     57s
  batch  40/127    112s
  batch  60/127    167s
  batch  80/127    222s
  batch 100/127    278s
  batch 120/127    333s
done in 348s


In [17]:
# sanity checks: right shapes, no empty rows, and similar photos get similar features
pooled = np.load(OUT / "feats_pooled.npy")
grid = np.load(OUT / "feats_grid.npy", mmap_mode="r")
print("pooled:", pooled.shape, pooled.dtype, "| grid:", grid.shape, grid.dtype)
print("all-zero rows:", int((np.abs(pooled.astype(np.float32)).sum(1) == 0).sum()))

v = pooled.astype(np.float32)
v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-8
q = img_index[df[df.split == "test"].image.iloc[0]]
nearest = np.argsort(-(v @ v[q]))[1:4]
print("\nquery image :", images[q], "->", df[df.image == images[q]].en.iloc[0])
for j in nearest:
    print("similar     :", images[j], "->", df[df.image == images[j]].en.iloc[0])

pooled: (8091, 2048) float16 | grid: (8091, 49, 2048) float16
all-zero rows: 0

query image : 1056338697_4f7d7ce270.jpg -> A blond woman in a blue shirt appears to wait for a ride .
similar     : 2385871317_44cde2f354.jpg -> A brown-haired girl wearing a yellow and pink shirt , and carrying a blue backpack is walking next to a blue wall .
similar     : 2724485630_7d2452df00.jpg -> Two blonde women are wearing white I love beer shirts .
similar     : 3544312930_3a0b8d70c1.jpg -> A girl looking away while holding a jacket .


**Expected:** `pooled: (8091, 2048) float16 | grid: (8091, 49, 2048) float16`, no all-zero rows, and the three "similar" images have captions about similar scenes (for example, all about dogs). That shows the features carry meaning.

## Save everything for the next notebooks

In [18]:
df.to_csv(OUT / "captions_split.csv", index=False, encoding="utf-8")
json.dump(images, open(OUT / "images.json", "w", encoding="utf-8"))
for lang in ["en", "bn"]:
    json.dump({"itos": vocab[lang], "max_words": MAX_WORDS[lang], "min_freq": MIN_FREQ},
              open(OUT / f"vocab_{lang}.json", "w", encoding="utf-8"), ensure_ascii=False)
    np.savez_compressed(OUT / f"encoded_{lang}.npz", ids=encoded[lang],
                        img_idx=df.image.map(img_index).to_numpy(np.int32),
                        split=df.split.to_numpy().astype("U6"))

for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(f"{p.name:24s} {p.stat().st_size / 1e6:8.1f} MB")

captions_split.csv           16.7 MB
encoded_bn.npz                0.6 MB
encoded_en.npz                0.6 MB
feats_grid.npy             1623.9 MB
feats_pooled.npy             33.1 MB
images.json                   0.2 MB
vocab_bn.json                 0.1 MB
vocab_en.json                 0.0 MB


**Expected files:** `captions_split.csv`, `images.json`, `vocab_en.json`, `vocab_bn.json`, `encoded_en.npz`, `encoded_bn.npz`, `feats_pooled.npy` (about 33 MB) and `feats_grid.npy` (about 1,600 MB).

**Next:** click **Save Version → Save & Run All (Commit)**. When the version finishes, notebook 02 can add this notebook's output as an input.

### Questions for the team
1. Why is the vocabulary built from training captions only?
2. What are `<start>`, `<end>`, `<pad>` and `<unk>` for?
3. Why is ResNet-50 frozen, and why do we save its output instead of running it during training?
4. What is the difference between the pooled vector and the 7 × 7 grid, and which model uses which?